In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
# Linalg
import jax.numpy as np
import numpy as onp

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm

# Data handling
import h5py as hf
import pandas as pd

# Analysis
from atomcorr.analysis.onsager_coefficients import OnsagerCoefficients

# Helper functions
from rich.progress import track

In [ ]:

main_prefixes = [
    "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_default_noise_strength_alignment_strength_scan_radius_1.0",
    # "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_default_noise_strength_alignment_strength_scan_radius_4.0",
    # "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_slow_noise_strength_alignment_strength_scan_radius_1.0",
    # "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_slow_noise_strength_alignment_strength_scan_radius_4.0"
]

In [ ]:
def normalize_covariance_matrix(covariance_matrix: np.ndarray):
    """
    Method for normalizing a covariance matrix.
    Returns
    -------
    normalized_covariance_matrix : np.ndarray
            A normalized covariance matrix, i.e, the matrix given if all of its inputs
            had been normalized.
    """
    order = np.shape(covariance_matrix)[0]

    diagonals = np.diagonal(covariance_matrix)

    repeated_diagonals = np.repeat(diagonals[None, :], order, axis=0)

    normalizing_matrix = np.sqrt(repeated_diagonals * repeated_diagonals.T)

    return covariance_matrix / normalizing_matrix

In [ ]:
params = np.linspace(0, 24, 25, dtype=int)
calculator = OnsagerCoefficients()
entropies = []
traces = []
corr_matrix = []

for main_prefix in main_prefixes:
    corr_data = pd.read_csv(f"{main_prefix}/aggregate.time_avg.ensemble_avg.lymburn_correlation_coefficient_test.csv")
    corr_matrix.append(
        corr_data["observable"].to_numpy()
    )
    for param in track(params):
        prefix = f"{main_prefix}/parameter-combination-{param}/seed-0"
        

        with hf.File(f"{prefix}/simulation_output_train.h5", "r") as db:
            data = db["agent_observables"]["agent_velocity"][:]

            full_correlation = calculator._compute_correlation_matrix(
                data, 
                data, 
                correlation_time=1000,
                data_range=300
            )
            eigs = np.clip(np.linalg.eigh(full_correlation)[0], 1e-8, None)
            eigs /= eigs.sum()

            entropy = (-eigs * np.log(eigs)).sum()
            entropies.append(entropy)
            traces.append(np.trace(full_correlation))

    


In [ ]:
noise_strength = [0.01, 0.1, 1.0, 10.0, 100.0]
alignment_strength = [0.001, 0.01, 0.1, 1.0, 10.0]

In [ ]:
x, y = np.meshgrid(noise_strength, alignment_strength)

In [ ]:
fig = plt.figure()
loss_ax = fig.add_subplot(projection='3d')
entropy_ax = fig.add_subplot(projection="3d")
trace_ax = fig.add_subplots(projection="3d")


entropy = np.reshape(np.array(entropies), (5, 5))
traces = np.reshape(np.array(traces), (5, 5))
corr_data = np.reshape(np.array(corr_matrix[0]), (5, 5))



In [ ]:
fig, ax = plt.subplots()

ax.plot(traces)
ax.set_yscale("log")

ax2 = ax.twinx()

ax2.plot(corr_matrix, "r")

ax.set_xlabel("parameter")


ax.set_ylabel("Trace")
ax2.set_ylabel("Correlation Coefficient")

# plt.savefig("traec-diagram.pdf")

plt.show()


In [ ]:
fig, ax = plt.subplots()

plt.scatter(traces, corr_matrix, c=entropies)

# plt.savefig("traec-diagram.pdf")
plt.xscale("log")

plt.colorbar()
plt.show()

In [ ]:
fig, ax = plt.subplots()

plt.scatter(entropies, corr_matrix, c=traces, norm=LogNorm())

# plt.savefig("traec-diagram.pdf")
plt.colorbar()
plt.show()

In [ ]:
fig, ax = plt.subplots()

plt.scatter(traces, entropies, c=corr_matrix)

# plt.savefig("traec-diagram.pdf")

plt.colorbar()
plt.xscale("log")
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(entropies)
ax2 = ax.twinx()

ax2.plot(corr_matrix, "r")

ax.set_xlabel("parameter")


ax.set_ylabel("Entropy")
ax2.set_ylabel("Correlation Coefficient")


# plt.savefig("entropy-diagram.pdf")
plt.show()
